# **Rate My Recipe**

## 1. Recipe Characteristics and Ratings

This notebook explores the relationship between recipe preparation time and user ratings.

The analysis focuses on preparation time categories and their representation in the Bayesian ranking.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from recipe_project.data_loader import load_recipes, load_interactions
from recipe_project.cleaning import filter_valid_interactions
from recipe_project.analysis import calculate_recipe_stats
from recipe_project.plots import preparation_time_rating_comparison

In [3]:
recipes = load_recipes()
interactions = load_interactions()

In [4]:
valid_interactions = filter_valid_interactions(interactions)
recipe_stats = calculate_recipe_stats(valid_interactions)

In [5]:
print("Valid interactions:", valid_interactions.shape)
print("Recipe statistics:", recipe_stats.shape)

display(recipe_stats.head())

Valid interactions: (1071520, 5)
Recipe statistics: (226590, 3)


,recipe_id,rating_count,average_rating
0,38,4,4.250000
1,39,1,3.000000
2,40,9,4.333333
3,41,2,4.500000
4,43,1,1.000000


## 2. Preparation Time

This section explores the distribution of recipe preparation times and their relationship with average user ratings.

### 2.1 Preparation Time Distribution

This section examines the distribution of recipe preparation times.

In [6]:
pd.set_option("display.float_format", "{:,.2f}".format)
recipes["minutes"].describe()

count         231,637.00
mean            9,398.55
std         4,461,963.04
min                 0.00
25%                20.00
50%                40.00
75%                65.00
max     2,147,483,647.00
Name: minutes, dtype: float64

In [7]:
print("Missing preparation times:", recipes["minutes"].isna().sum())

print("Zero preparation times:", (recipes["minutes"] == 0).sum())

print("Negative preparation times:", (recipes["minutes"] < 0).sum())

print("Preparation times above 1,000 minutes:", (recipes["minutes"] > 1000).sum())

print("Most frequent preparation times:")
print(recipes["minutes"].value_counts().head(10))

Missing preparation times: 0
Zero preparation times: 1094
Negative preparation times: 0
Preparation times above 1,000 minutes: 2240
Most frequent preparation times:
minutes
30    18391
20    14822
40    13772
25    13204
35    12385
15    12213
45    11517
10    11291
60    10302
50     9929
Name: count, dtype: int64


In [8]:
print(
    recipes.loc[
        recipes["minutes"] > 1000,
        ["id", "name", "minutes"]
    ].sort_values("minutes", ascending=False).head(15)
)

            id                                             name     minutes
144074  261647                            no bake granola balls  2147483647
109624  447963                        how to preserve a husband     1051200
106563  291571                           homemade fruit liquers      288000
38116   216215    celtic druid s honey mead   meade   metheglin      259260
106963  425681                                 homemade vanilla      259205
155044   98912                                     peach brandy      216015
5497     70551                                 angelica liqueur      201610
177906   12931                                          rumtopf      187200
170999  172114                                raspberry liqueur      172810
53721   153634                                 cinnamon extract      172800
102299    2714                                 hard apple cider      167040
171791  236340  recipe for elephant stew  from griot s cookbook      146880
41471    133

### Data Cleaning: Preparation Time

The preparation time data contains 1,094 recipes with zero minutes and 2,240 recipes with values above 1,000 minutes. These values may distort the statistical analysis and visualizations.

For this analysis, we exclude recipes with preparation times equal to zero or greater than 1,000 minutes. The threshold is an operational choice intended to limit the influence of extreme values, not proof that all excluded preparation times are incorrect.

The original dataset remains unchanged. A filtered copy is used for the preparation time analysis.

### 2.2 Exclusion of Anomalous Values

In [9]:
recipes_time = recipes[
    (recipes["minutes"] > 0) &
    (recipes["minutes"] <= 1000)
].copy()

### 2.3 Preparation Time Categories

Recipes are grouped into four categories:

- **Quick:** up to 30 minutes.
- **Medium:** 31–60 minutes.
- **Long:** 61–90 minutes.
- **Very Long:** over 90 minutes.

This classification allows us to compare average ratings across different preparation times.

In [10]:
preparation_categories = []

for minutes in recipes_time["minutes"]:
    if pd.isna(minutes) or minutes < 0:
        preparation_categories.append("Unknown")
    elif minutes <= 30:
        preparation_categories.append("Quick")
    elif minutes <= 60:
        preparation_categories.append("Medium")
    elif minutes <= 90:
        preparation_categories.append("Long")
    else:
        preparation_categories.append("Very Long")

recipes_time["preparation_category"] = preparation_categories

recipes_time["preparation_category"].value_counts()

preparation_category
Quick        97959
Medium       70077
Very Long    31196
Long         29071
Name: count, dtype: int64

### 2.4 Preparation Time and Ratings
This section combines recipe preparation times with user rating statistics. Recipes are grouped by preparation time category to compare their average ratings and the number of ratings received.

In [11]:
recipe_analysis = recipes_time.merge(
    recipe_stats,
    left_on="id",
    right_on="recipe_id",
    how="inner"
)

In [12]:
category_stats = recipe_analysis.groupby("preparation_category").agg(
    recipe_count=("id", "count"),
    average_rating=("average_rating", "mean"),
    average_rating_count=("rating_count", "mean")
)

display(category_stats)

,recipe_count,average_rating,average_rating_count
preparation_category,,,
Long,28352,4.58,4.86
Medium,68495,4.56,4.75
Quick,96190,4.60,4.57
Very Long,30346,4.57,5.08


### 2.5 Preparation Time and Bayesian Ranking

In [16]:
from recipe_project.ranking import bayesian_weighted_ranking

weighted_ranking = bayesian_weighted_ranking(recipe_stats)

In [18]:
ranking_by_category = weighted_ranking.merge(
    recipes_time[["id", "preparation_category"]],
    left_on="recipe_id",
    right_on="id",
    how="inner"
)

In [19]:
top_100 = ranking_by_category.nlargest(100, "bayesian_score")

category_distribution_100 = (
    top_100["preparation_category"].value_counts()
)

display(category_distribution_100)

preparation_category
Quick        43
Medium       25
Long         16
Very Long    16
Name: count, dtype: int64

**Observations**

Quick recipes are the most represented in the 100 Bayesian ranking. However, this result should be interpreted considering the different sizes of the preparation time categories.

In [20]:
overall_distribution = (
    ranking_by_category["preparation_category"]
    .value_counts(normalize=True) * 100
)

display(overall_distribution.round(2))

preparation_category
Quick       43.06
Medium      30.66
Very Long   13.58
Long        12.69
Name: proportion, dtype: float64

**Observations**

- Quick recipes represent 43% of the top 100, almost identical to their 42.92% share in the overall ranking.
- Medium recipes are less represented in the top 100 (25% vs. 30.23%).
- Long and Very Long recipes are slightly more represented in the top 100.

**Conclusion**

Preparation time does not appear to be strongly associated with the representation of recipes in the top 100 Bayesian ranking. Quick recipes have a similar share in the top 100 compared to their overall distribution, while Long and Very Long recipes are slightly more represented. These differences are relatively small and should be interpreted with caution.

# 3. Ingrediente Analysis
This section investigates whether the number of ingredients in a recipe is associated with differences in user ratings and recipe rankings.

The analysis considers the distribution of ingredient counts, compares average ratings across ingredient categories, and examines how these categories are represented in the Bayesian Weighted Ranking.

### 3.1 Ingredient Count Distribution

In [21]:
recipes["n_ingredients"].describe()

count   231,637.00
mean          9.05
std           3.73
min           1.00
25%           6.00
50%           9.00
75%          11.00
max          43.00
Name: n_ingredients, dtype: float64

### 3.2 Ingredient Count Categories

Recipes are divided into three categories according to their number of ingredients:

- **Few Ingredients:** 1–4 ingredients.
- **Medium Ingredients:** 5–10 ingredients.
- **Many Ingredients:** more than 10 ingredients.

This classification provides a simple way to compare recipes with different levels of ingredient complexity.

In [22]:
ingredient_counts = recipes["n_ingredients"]

print("Missing values:", ingredient_counts.isna().sum())
print("Non-numeric values:", pd.to_numeric(
    ingredient_counts, errors="coerce"
).isna().sum() - ingredient_counts.isna().sum())
print("Non-integer values:", (
    ingredient_counts.dropna() % 1 != 0
).sum())
print("Non-positive values:", (
    ingredient_counts.dropna() <= 0
).sum())

Missing values: 0
Non-numeric values: 0
Non-integer values: 0
Non-positive values: 0


In [23]:
ingredient_categories = []

for n in recipes["n_ingredients"]:
    if n <= 4:
        ingredient_categories.append("Few Ingredients")
    elif n <= 10:
        ingredient_categories.append("Medium Ingredients")
    else:
        ingredient_categories.append("Many Ingredients")

recipes_ingredients = recipes.copy()
recipes_ingredients["ingredient_category"] = ingredient_categories

recipes_ingredients["ingredient_category"].value_counts()

ingredient_category
Medium Ingredients    137217
Many Ingredients       72818
Few Ingredients        21602
Name: count, dtype: int64

### 3.3 Ingredient Count and Ratings

This section combines ingredient count categories with user rating statistics. Recipes are grouped by ingredient category to compare their average ratings and the number of ratings received.

In [24]:
recipe_ingredient_analysis = recipes_ingredients.merge(
    recipe_stats,
    left_on="id",
    right_on="recipe_id",
    how="inner"
)

In [25]:
ingredient_category_stats = recipe_ingredient_analysis.groupby(
    "ingredient_category"
).agg(
    recipe_count=("id", "count"),
    average_rating=("average_rating", "mean"),
    average_rating_count=("rating_count", "mean")
)

display(ingredient_category_stats)

,recipe_count,average_rating,average_rating_count
ingredient_category,,,
Few Ingredients,21136,4.62,4.83
Many Ingredients,71146,4.58,4.55
Medium Ingredients,134308,4.57,4.81


### Results and Interpretation

The average ratings are very similar across the three ingredient categories, ranging from 4.57 to 4.62.

Recipes with Few Ingredients have the highest average rating (4.62), while recipes with Medium Ingredients have the lowest (4.57). Recipes with Many Ingredients show an intermediate average rating (4.58).

These results suggest that the number of ingredients is not strongly associated with differences in average recipe ratings in this dataset. However, the differences are small and should not be interpreted as evidence of a causal relationship.

### 3.4 Ingredient Count and Bayesian Ranking

In [26]:
ranking_by_ingredient = weighted_ranking.merge(
    recipes_ingredients[["id", "ingredient_category"]],
    left_on="recipe_id",
    right_on="id",
    how="inner"
)

In [27]:
top_100_ingredients = ranking_by_ingredient.nlargest(
    100, "bayesian_score"
)

ingredient_distribution_100 = (
    top_100_ingredients["ingredient_category"].value_counts()
)

display(ingredient_distribution_100)

ingredient_category
Medium Ingredients    49
Many Ingredients      29
Few Ingredients       22
Name: count, dtype: int64

In [28]:
overall_ingredient_distribution = (
    ranking_by_ingredient["ingredient_category"]
    .value_counts(normalize=True) * 100
)

display(overall_ingredient_distribution.round(2))

ingredient_category
Medium Ingredients   59.27
Many Ingredients     31.40
Few Ingredients       9.33
Name: proportion, dtype: float64

### Observations

- Recipes with Few Ingredients are more represented in the top 100 (22% compared to 9.33% in the overall ranking).
- Recipes with Medium Ingredients are less represented (49% compared to 59.27%).
- Recipes with Many Ingredients are also slightly less represented (29% compared to 31.40%).

### Conclusion

Recipes with Few Ingredients appear more frequently in the top 100 of the Bayesian Weighted Ranking compared to their share in the overall ranking. This suggests that recipes with fewer ingredients are more represented among the highest-ranked recipes. However, this result does not establish a causal relationship between the number of ingredients and user preferences.